In [4]:
import os
import glob
import pandas as pd
import torch
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:


tsv_files = glob.glob(
    "/kaggle/input/**/*.tsv",
    recursive=True
)

print("TSV files found:")

for file in tsv_files:
    print(file)

if len(tsv_files) == 0:
    raise FileNotFoundError(
        "No TSV file found inside /kaggle/input/"
    )

DATA_PATH = tsv_files[0]

print("\nUsing dataset:")
print(DATA_PATH)

TSV files found:
/kaggle/input/datasets/robinhossain231/microsoft/blp23_sentiment_dev.tsv

Using dataset:
/kaggle/input/datasets/robinhossain231/microsoft/blp23_sentiment_dev.tsv


In [8]:

df = pd.read_csv(
    DATA_PATH,
    sep="\t"
)

print("Dataset loaded successfully!")

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully!

Shape:
(3934, 3)

Columns:
['id', 'text', 'label']


,id,text,label
0,5300,নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়ত...,Negative
1,15392,বিদেশে পড়ালেখা করছে বাংলাদেশের প্রচুর ছেলেময়ের...,Positive
2,6904,মাননীয় আপনি নিজে না বলে সস্তায় কোনো মন্ত্রী কে...,Negative
3,30790,* করোনার টিকা নিলেন বিএনপি চেয়ারপারসন বেগম খ...,Positive
4,2770,একজন প্রধানমন্ত্রীর এমন বক্তব্য জাতির জন্য লজ্...,Negative


In [9]:


TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

# Model will only use the text column
sentiment_df = df[["id", TEXT_COLUMN, LABEL_COLUMN]].copy()

# Keep original structure
sentiment_df["text"] = sentiment_df["text"].astype(str)
sentiment_df["label"] = (
    sentiment_df["label"]
    .astype(str)
    .str.strip()
)

print("Dataset prepared successfully!")

print("\nShape:")
print(sentiment_df.shape)

print("\nColumns:")
print(sentiment_df.columns.tolist())

display(sentiment_df.head())

Dataset prepared successfully!

Shape:
(3934, 3)

Columns:
['id', 'text', 'label']


,id,text,label
0,5300,নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়ত...,Negative
1,15392,বিদেশে পড়ালেখা করছে বাংলাদেশের প্রচুর ছেলেময়ের...,Positive
2,6904,মাননীয় আপনি নিজে না বলে সস্তায় কোনো মন্ত্রী কে...,Negative
3,30790,* করোনার টিকা নিলেন বিএনপি চেয়ারপারসন বেগম খ...,Positive
4,2770,একজন প্রধানমন্ত্রীর এমন বক্তব্য জাতির জন্য লজ্...,Negative


In [10]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [11]:
def generate_response(prompt, max_new_tokens=50):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [14]:
import re

def extract_label(response):
    text = response.strip()

    # For CoT output
    match = re.search(
        r"final\s*label\s*:\s*(positive|negative|neutral)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1).capitalize()

    # For Zero-Shot / Few-Shot output
    matches = re.findall(
        r"\b(positive|negative|neutral)\b",
        text,
        re.IGNORECASE
    )

    if matches:
        return matches[-1].capitalize()

    return "Unknown"

In [15]:
def zero_shot_prompt(text):
    return f"""
You are a Bengali sentiment analysis model.

Analyze the sentiment of the following Bengali text.

Text:
{text}

Classify the sentiment as one of:
Positive
Negative
Neutral

Return only one word:
Positive, Negative, or Neutral.
"""

In [16]:
test_text = sentiment_df.iloc[0]["text"]

prompt = zero_shot_prompt(test_text)

response = generate_response(
    prompt,
    max_new_tokens=10
)

prediction = extract_label(response)

print("Text:")
print(test_text)

print("\nModel Response:")
print(response)

print("\nPredicted Sentiment:")
print(prediction)

Text:
নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়তারা শুরু করছে । 

Model Response:
Negative

The text expresses a negative sentiment by suggesting

Predicted Sentiment:
Negative


In [17]:
zero_shot_predictions = []

for i, text in enumerate(sentiment_df["text"]):

    prompt = zero_shot_prompt(text)

    response = generate_response(
        prompt,
        max_new_tokens=10
    )

    prediction = extract_label(response)

    zero_shot_predictions.append(prediction)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(sentiment_df)}")

Processed 100/3934
Processed 200/3934
Processed 300/3934
Processed 400/3934
Processed 500/3934
Processed 600/3934
Processed 700/3934
Processed 800/3934
Processed 900/3934
Processed 1000/3934
Processed 1100/3934
Processed 1200/3934
Processed 1300/3934
Processed 1400/3934
Processed 1500/3934
Processed 1600/3934
Processed 1700/3934
Processed 1800/3934
Processed 1900/3934
Processed 2000/3934
Processed 2100/3934
Processed 2200/3934
Processed 2300/3934
Processed 2400/3934
Processed 2500/3934
Processed 2600/3934
Processed 2700/3934
Processed 2800/3934
Processed 2900/3934
Processed 3000/3934
Processed 3100/3934
Processed 3200/3934
Processed 3300/3934
Processed 3400/3934
Processed 3500/3934
Processed 3600/3934
Processed 3700/3934
Processed 3800/3934
Processed 3900/3934


In [19]:
sentiment_df["zero_shot_prediction"] = zero_shot_predictions

display(
    sentiment_df[
        ["id", "text", "label", "zero_shot_prediction"]
    ].head(10)
)

,id,text,label,zero_shot_prediction
0,5300,নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়ত...,Negative,Negative
1,15392,বিদেশে পড়ালেখা করছে বাংলাদেশের প্রচুর ছেলেময়ের...,Positive,Positive
2,6904,মাননীয় আপনি নিজে না বলে সস্তায় কোনো মন্ত্রী কে...,Negative,Negative
3,30790,* করোনার টিকা নিলেন বিএনপি চেয়ারপারসন বেগম খ...,Positive,Neutral
4,2770,একজন প্রধানমন্ত্রীর এমন বক্তব্য জাতির জন্য লজ্...,Negative,Negative
5,15617,শিক্ষক দের মান উন্নয়নে ও কাজ করতে হবে,Positive,Neutral
6,11628,চোর বেশি তাই,Negative,Neutral
7,sentinob_1080,"হাহাহাহা , না আপনার ভয়ের কোন কারণ নেই । সবগুলো...",Neutral,Positive
8,27109,সেসময় একটু বেছে নিতে হবে আপনার সময় ও পছন্দ অ...,Neutral,Neutral
9,30142,সম্মেলন আয়োজনকে ঘিরে হেফাজতে ইসলামের নেতৃত্বে...,Negative,Negative


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score
)

# Ground Truth
y_true = sentiment_df["label"].str.lower().str.strip()

# Zero-Shot Prediction
y_pred = sentiment_df["zero_shot_prediction"].str.lower().str.strip()

# Metrics
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

kappa = cohen_kappa_score(y_true, y_pred)

# Display Results
print("===== Zero-Shot Sentiment Analysis =====")
print(f"Accuracy:        {accuracy:.4f}")
print(f"Precision:       {precision:.4f}")
print(f"Recall:          {recall:.4f}")
print(f"F1-Score:       {f1:.4f}")
print(f"Cohen's Kappa:  {kappa:.4f}")

===== Zero-Shot Sentiment Analysis =====
Accuracy:        0.4512
Precision:       0.5816
Recall:          0.4512
F1-Score:       0.4580
Cohen's Kappa:  0.2097


In [22]:
def few_shot_prompt(text):
    return f"""
You are a Bengali sentiment analysis model.

Use the following labeled examples to understand the sentiment classification task.

Example 1:
Text: এই পণ্যটি খুব ভালো এবং আমি এটি পছন্দ করেছি।
Sentiment: Positive

Example 2:
Text: সেবাটি খুব খারাপ, আমি একদম সন্তুষ্ট নই।
Sentiment: Negative

Example 3:
Text: পণ্যটি মোটামুটি, বিশেষ ভালো বা খারাপ কিছু নয়।
Sentiment: Neutral

Now classify the sentiment of the following Bengali text.

Text:
{text}

Choose exactly one label:
Positive
Negative
Neutral

Return only one word:
Positive, Negative, or Neutral.
"""

In [ ]:
few_shot_predictions = []

for i, text in enumerate(sentiment_df["text"]):

    prompt = few_shot_prompt(text)

    response = generate_response(
        prompt,
        max_new_tokens=10
    )

    prediction = extract_label(response)

    few_shot_predictions.append(prediction)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(sentiment_df)}")

sentiment_df["few_shot_prediction"] = few_shot_predictions

print("Few-Shot prediction completed.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score
)

y_true = sentiment_df["label"].str.lower().str.strip()
y_pred = sentiment_df["few_shot_prediction"].str.lower().str.strip()

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

kappa = cohen_kappa_score(y_true, y_pred)

print("===== Qwen Few-Shot Sentiment Analysis =====")
print(f"Accuracy:       {accuracy:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"F1-Score:       {f1:.4f}")
print(f"Cohen's Kappa:  {kappa:.4f}")